In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.tensorboard import SummaryWriter
import os, json


In [21]:
# Load data
df = pd.read_csv("data/KaggleV2-May-2016.csv")

# Target: Yes/No -> 1/0
df['No_show'] = df['No-show'].map({'No': 0, 'Yes': 1})

# Removing the  impossible ages if any exists
df = df[df['Age'] >= 0].copy()

print(df.shape)
print(df['No_show'].value_counts(normalize=True))
df.head()


(110526, 15)
No_show
0    0.798066
1    0.201934
Name: proportion, dtype: float64


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,No_show
0,2.990000e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No,0
1,5.590000e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No,0
2,4.260000e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No,0
3,8.680000e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No,0
4,8.840000e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No,0


In [22]:
df['ScheduledDay']   = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

df['waiting_days'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days.clip(lower=0)
df['appointment_weekday'] = df['AppointmentDay'].dt.dayofweek

df[['waiting_days', 'appointment_weekday']].describe()


,waiting_days,appointment_weekday
count,110526.000000,110526.000000
mean,9.532825,1.858260
std,15.027769,1.371667
min,0.000000,0.000000
25%,0.000000,1.000000
50%,3.000000,2.000000
75%,14.000000,3.000000
max,178.000000,5.000000


In [23]:
feature_cols = [
    'Age',
    'Scholarship',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'SMS_received',
    'waiting_days',
    'appointment_weekday'
]

X = df[feature_cols].astype(float)
y = df['No_show'].astype(float)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, y_train.shape)
print("Val:",   X_val_scaled.shape,   y_val.shape)
print("Test:",  X_test_scaled.shape,  y_test.shape)


X shape: (110526, 9)
y shape: (110526,)
Train: (88420, 9) (88420,)
Val: (11053, 9) (11053,)
Test: (11053, 9) (11053,)


In [ ]:
device = "cpu"
print("Using device:", device)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train.values.reshape(-1,1), dtype=torch.float32).to(device)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val.values.reshape(-1,1), dtype=torch.float32).to(device)


Using device: cpu


In [ ]:
class NoShowNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = NoShowNet(X_train_scaled.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model


NoShowNet(
  (net): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
    (6): ReLU()
  )
)

In [31]:
batch_X = X_train_t[:128]
batch_y = y_train_t[:128]

for epoch in range(50):
    optimizer.zero_grad()
    out = model(batch_X)
    loss = criterion(out, batch_y)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"sanity epoch {epoch:02d} | loss {loss.item():.4f}")


sanity epoch 00 | loss 23.5782
sanity epoch 10 | loss 3.0407
sanity epoch 20 | loss 0.5954


RuntimeError: all elements of input should be between 0 and 1

In [28]:
writer = SummaryWriter(log_dir="logs/run1")

for epoch in range(10):
    # train on full train set
    model.train()
    optimizer.zero_grad()
    out = model(X_train_t)
    loss = criterion(out, y_train_t)
    loss.backward()
    optimizer.step()

    # validation
    model.eval()
    with torch.no_grad():
        val_out = model(X_val_t)
        val_loss = criterion(val_out, y_val_t)

    writer.add_scalar("loss/train", loss.item(), epoch)
    writer.add_scalar("loss/val",   val_loss.item(), epoch)
    print(f"epoch {epoch:02d} | train {loss.item():.4f} | val {val_loss.item():.4f}")

writer.close()

# save model + config
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/no_show_checkpoint.pt")

config = {"lr": 1e-3, "batch_size": "full", "epochs": 10, "optimizer": "Adam"}
with open("logs/config.json","w") as f:
    json.dump(config, f)


epoch 00 | train 0.4921 | val 0.4867
epoch 01 | train 0.4915 | val 0.4859
epoch 02 | train 0.4905 | val 0.4851
epoch 03 | train 0.4897 | val 0.4845
epoch 04 | train 0.4894 | val 0.4839
epoch 05 | train 0.4888 | val 0.4834
epoch 06 | train 0.4883 | val 0.4830
epoch 07 | train 0.4880 | val 0.4827
epoch 08 | train 0.4876 | val 0.4823
epoch 09 | train 0.4877 | val 0.4820


In [29]:
import os

print("Logs dir:", os.listdir("logs"))
print("Models dir:", os.listdir("models"))


Logs dir: ['config.json', 'run1']
Models dir: ['no_show_checkpoint.pt']
